In [2]:
from pathlib import Path

In [3]:
cwd = Path.cwd()
print(f"Working directory: {cwd}")

Working directory: /Users/karoevovo/lab-04-reading-and-writing-data-files/data/raw


In [4]:
files = list(cwd.rglob("*.csv")) + list(cwd.rglob("*.json"))
print(f"\nData files found:")
for f in files:
    print(f"  {f.relative_to(cwd)}")


Data files found:
  beanscene_menu.csv
  .ipynb_checkpoints/beanscene_menu-checkpoint.csv
  beanscene_config.json
  .ipynb_checkpoints/beanscene_config-checkpoint.json


In [5]:
import csv

In [6]:
menu = []

with open("beanscene_menu.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        menu.append({
            "name":       row["name"],
            "price":      float(row["price"]),
            "units_sold": int(row["units_sold"]),
            "category":   row["category"]
        })

In [7]:
print(menu[0])
print("price type:", type(menu[0]["price"]))
print("units_sold type:", type(menu[0]["units_sold"]))
print(f"Total items loaded: {len(menu)}")

{'name': 'Espresso', 'price': 3.5, 'units_sold': 42, 'category': 'Hot Drinks'}
price type: <class 'float'>
units_sold type: <class 'int'>
Total items loaded: 8


In [8]:
import json

with open("beanscene_config.json", encoding="utf-8") as f:
    config = json.load(f)

print(f"Café name:               {config['cafe_name']}")
print(f"Analysis week:           {config['analysis_week']}")
print(f"High threshold:          {config['performance_thresholds']['high']}")
print(f"Medium threshold:        {config['performance_thresholds']['medium']}")
print(f"Low performer threshold: {config['low_performer_threshold']}")

Café name:               BeanScene
Analysis week:           2024-W04
High threshold:          250.0
Medium threshold:        150.0
Low performer threshold: 150.0


In [9]:
def classify_item(revenue, thresholds):
    if revenue >= thresholds["high"]:
        return "High Performer"
    elif revenue >= thresholds["medium"]:
        return "Steady"
    else:
        return "Needs Attention"



In [10]:
thresholds = config["performance_thresholds"]

for item in menu:
    item["revenue"]        = round(item["price"] * item["units_sold"], 2)
    item["classification"] = classify_item(item["revenue"], thresholds)


In [11]:
print(f"{'Name':<18} {'Price':>6} {'Units':>6} {'Revenue':>9}  {'Classification'}")
print("-" * 60)
for item in menu:
    print(f"{item['name']:<18} {item['price']:>6.2f} {item['units_sold']:>6} "
          f"{item['revenue']:>9.2f}  {item['classification']}")

Name                Price  Units   Revenue  Classification
------------------------------------------------------------
Espresso             3.50     42    147.00  Needs Attention
Latte                4.75     61    289.75  High Performer
Cappuccino           4.50     38    171.00  Steady
Cold Brew            5.00     29    145.00  Needs Attention
Flat White           4.25     47    199.75  Steady
Matcha Latte         5.50     33    181.50  Steady
Iced Americano       4.00     55    220.00  Steady
Chai Latte           4.75     41    194.75  Steady


In [12]:
import csv
from pathlib import Path


In [13]:
output_dir = Path("data") / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "menu_results.csv"

with open(output_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "revenue", "classification"])
    writer.writeheader()
    for item in menu:
        writer.writerow({
            "name":           item["name"],
            "revenue":        item["revenue"],
            "classification": item["classification"]
        })

In [14]:
print(f"Results saved to: {output_path}")
print(f"Rows written: {len(menu)}")


Results saved to: data/processed/menu_results.csv
Rows written: 8


In [15]:
import json
from pathlib import Path

In [16]:
total_revenue = round(sum(item["revenue"] for item in menu), 2)
top_performer = max(menu, key=lambda item: item["revenue"])
needs_attention = [item["name"] for item in menu if item["classification"] == "Needs Attention"]

summary = {
    "cafe_name":        config["cafe_name"],
    "analysis_week":    config["analysis_week"],
    "total_revenue":    total_revenue,
    "total_items":      len(menu),
    "top_performer":    top_performer["name"],
    "needs_attention":  needs_attention
}

In [17]:
output_path = Path("data") / "processed" / "weekly_summary.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

print(f"Summary saved to: {output_path}")
print(json.dumps(summary, indent=4))

Summary saved to: data/processed/weekly_summary.json
{
    "cafe_name": "BeanScene",
    "analysis_week": "2024-W04",
    "total_revenue": 1548.75,
    "total_items": 8,
    "top_performer": "Latte",
    "needs_attention": [
        "Espresso",
        "Cold Brew"
    ]
}


In [18]:
import csv
import json
from pathlib import Path


In [19]:
print("=== menu_results.csv ===")
with open(Path("data") / "processed" / "menu_results.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row)

=== menu_results.csv ===
{'name': 'Espresso', 'revenue': '147.0', 'classification': 'Needs Attention'}
{'name': 'Latte', 'revenue': '289.75', 'classification': 'High Performer'}
{'name': 'Cappuccino', 'revenue': '171.0', 'classification': 'Steady'}
{'name': 'Cold Brew', 'revenue': '145.0', 'classification': 'Needs Attention'}
{'name': 'Flat White', 'revenue': '199.75', 'classification': 'Steady'}
{'name': 'Matcha Latte', 'revenue': '181.5', 'classification': 'Steady'}
{'name': 'Iced Americano', 'revenue': '220.0', 'classification': 'Steady'}
{'name': 'Chai Latte', 'revenue': '194.75', 'classification': 'Steady'}


In [20]:
print("\n=== weekly_summary.json ===")
with open(Path("data") / "processed" / "weekly_summary.json", encoding="utf-8") as f:
    summary = json.load(f)
    print(json.dumps(summary, indent=4))


=== weekly_summary.json ===
{
    "cafe_name": "BeanScene",
    "analysis_week": "2024-W04",
    "total_revenue": 1548.75,
    "total_items": 8,
    "top_performer": "Latte",
    "needs_attention": [
        "Espresso",
        "Cold Brew"
    ]
}
